# Null Space Analysis

This notebook will extract and analyze neuron spiking data from NWB files.


In [ ]:
import os
from pathlib import Path
from pynwb import NWBHDF5IO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set up paths
data_root = Path("/Users/jundazhu/SBCAT/000469")


## All Functions

All functions are defined below, organized by category:
- **Trial Selection**: Select trials by load, correctness, probe in/out, or indices
- **File Listing**: List and select NWB files
- **Data Extraction**: Extract spike data and trial information from NWB files
- **DataFrame Creation**: Convert extraction results to DataFrames
- **Spike Segmentation**: Segment spikes by trials, aligned to specific events
- **Workflow**: Process single or multiple sessions

In [ ]:
def select_trials_by_load(trials_df, load_values):
    """
    Select trials by memory load values (session-level).
    Args:
        trials_df: DataFrame with trial info
        load_values: int or list of int (e.g., 1, or [1,3])
    Returns:
        selected_trials_df: Filtered DataFrame
        selected_indices: np.array of row indices selected
    """
    if 'loads' not in trials_df.columns:
        raise ValueError("trials_df must have a 'loads' column")
    if not isinstance(load_values, (list, tuple, np.ndarray)):
        load_values = [load_values]
    mask = trials_df['loads'].isin(load_values)
    selected_indices = np.where(mask)[0]
    selected_trials_df = trials_df.iloc[selected_indices].copy()
    return selected_trials_df, selected_indices

def select_trials_by_correctness(trials_df, correct_only=True):
    """
    Select trials by correctness.
    Args:
        trials_df: DataFrame
        correct_only: bool
    Returns:
        selected_trials_df: Filtered DataFrame
        selected_indices: np.array of selected
    """
    if 'response_accuracy' not in trials_df.columns:
        raise ValueError("trials_df must have a 'response_accuracy' column")
    mask = (trials_df['response_accuracy'] == 1) if correct_only else (trials_df['response_accuracy'] != 1)
    selected_indices = np.where(mask)[0]
    selected_trials_df = trials_df.iloc[selected_indices].copy()
    return selected_trials_df, selected_indices

def select_trials_by_probe_in_out(trials_df, probe_in_out):
    """
    Select trials by probe in/out status.
    Args:
        trials_df: DataFrame
        probe_in_out: 'in' or 'out'
    Returns:
        selected_trials_df: Filtered DataFrame
        selected_indices: np.array of indices
    """
    if 'probe_in_out' not in trials_df.columns:
        raise ValueError("trials_df must have a 'probe_in_out' column")
    mask = (trials_df['probe_in_out'] == probe_in_out)
    selected_indices = np.where(mask)[0]
    selected_trials_df = trials_df.iloc[selected_indices].copy()
    return selected_trials_df, selected_indices

def select_trials_by_indices(trials_df, trial_indices):
    """
    Select trials by explicit trial indices.
    Args:
        trials_df: DataFrame
        trial_indices: list/array of 0-based trial indices
    Returns:
        selected_trials_df: Filtered DataFrame
        selected_indices: np.array of indices
    """
    selected_indices = np.array(trial_indices)
    selected_trials_df = trials_df.iloc[selected_indices].copy()
    return selected_trials_df, selected_indices

def list_available_files(data_dir, session_filter=None):
    """List all available NWB files with indices for selection.
    Args:
        data_dir: path object or string to root dir
        session_filter: 'ses-1', 'ses-2', or None
    Returns:
        files: list of Path
    """
    files = sorted(data_dir.rglob("*.nwb"))
    if session_filter in ["ses-1", "ses-2"]:
        files = [f for f in files if session_filter in f.name]
        print(f"Found {len(files)} NWB files (session filter = {session_filter}):\n")
    else:
        print(f"Found {len(files)} NWB files:\n")
    for i, f in enumerate(files):
        rel_path = f.relative_to(data_dir)
        print(f"{i:3d}. {rel_path}")
    return files

def extract_spike_data(filepath, include_waveforms=False, quality_filters=None, include_trials=True):
    """
    Extract spike data from an NWB file.
    Args:
        filepath: Path object or string for file.
        include_waveforms: bool
        quality_filters: dict
        include_trials: bool
    Returns:
        dict with
          - 'filepath'
          - 'subject_id'
          - 'session_id'
          - 'units': list of dict
          - 'trials': DataFrame or None
    """
    filepath = Path(filepath)
    # Subject/session from file name
    parts = filepath.stem.split('_')
    subject_id = None
    session_id = None
    for part in parts:
        if part.startswith('sub-'):
            subject_id = int(part.split('-')[1])
        elif part.startswith('ses-'):
            session_id = int(part.split('-')[1])
    result = {
        'filepath': str(filepath),
        'subject_id': subject_id,
        'session_id': session_id,
        'units': []
    }
    try:
        with NWBHDF5IO(str(filepath), mode="r", load_namespaces=True) as io:
            nwbf = io.read()
            # trials
            if include_trials and nwbf.trials is not None:
                try:
                    trials_df = nwbf.trials.to_dataframe()
                    result['trials'] = trials_df
                except Exception as e:
                    print(f"Warning: Could not extract trials from {filepath.name}: {e}")
                    result['trials'] = None
            else:
                result['trials'] = None
            if nwbf.units is None:
                print(f"Warning: No units found in {filepath.name}")
                return result
            units_df = nwbf.units.to_dataframe()
            electrodes_df = nwbf.electrodes.to_dataframe() if nwbf.electrodes is not None else None
            for unit_idx in units_df.index:
                unit_data = {
                    'unit_id': int(unit_idx),
                    'spike_times': None,
                    'electrode_idx': None,
                    'cluster_id': None,
                    'mean_snr': None,
                    'peak_snr': None,
                    'isolation_distance': None,
                    'mean_proj_dist': None,
                    'n_spikes': 0,
                    'brain_region': None
                }
                # spike times
                try:
                    spike_times = np.array(nwbf.units['spike_times'][unit_idx][:])
                    unit_data['spike_times'] = spike_times
                    unit_data['n_spikes'] = len(spike_times)
                except Exception as e:
                    print(f"Warning: Could not extract spike times for unit {unit_idx}: {e}")
                    continue
                # electrode info
                if 'electrodes' in units_df.columns:
                    try:
                        electrode_info = units_df.loc[unit_idx, 'electrodes']
                        if hasattr(electrode_info, 'index'):
                            unit_data['electrode_idx'] = int(electrode_info.index[0]) if len(electrode_info.index) > 0 else None
                        else:
                            unit_data['electrode_idx'] = int(electrode_info) if pd.notna(electrode_info) else None
                        if unit_data['electrode_idx'] is not None and electrodes_df is not None:
                            if 'location' in electrodes_df.columns:
                                unit_data['brain_region'] = electrodes_df.loc[unit_data['electrode_idx'], 'location']
                    except Exception:
                        pass
                # quality metrics
                if 'clusterID_orig' in units_df.columns:
                    unit_data['cluster_id'] = units_df.loc[unit_idx, 'clusterID_orig']
                if 'waveforms_mean_snr' in units_df.columns:
                    unit_data['mean_snr'] = units_df.loc[unit_idx, 'waveforms_mean_snr']
                if 'waveforms_peak_snr' in units_df.columns:
                    unit_data['peak_snr'] = units_df.loc[unit_idx, 'waveforms_peak_snr']
                if 'waveforms_isolation_distance' in units_df.columns:
                    unit_data['isolation_distance'] = units_df.loc[unit_idx, 'waveforms_isolation_distance']
                if 'waveforms_mean_proj_dist' in units_df.columns:
                    unit_data['mean_proj_dist'] = units_df.loc[unit_idx, 'waveforms_mean_proj_dist']
                # filter
                if quality_filters is not None:
                    skip_unit = False
                    if 'min_snr' in quality_filters and unit_data['mean_snr'] is not None:
                        if unit_data['mean_snr'] < quality_filters['min_snr']:
                            skip_unit = True
                    if 'min_isolation_distance' in quality_filters and unit_data['isolation_distance'] is not None:
                        if unit_data['isolation_distance'] < quality_filters['min_isolation_distance']:
                            skip_unit = True
                    if 'min_n_spikes' in quality_filters:
                        if unit_data['n_spikes'] < quality_filters['min_n_spikes']:
                            skip_unit = True
                    if skip_unit:
                        continue
                # waveforms
                if include_waveforms and 'waveforms' in units_df.columns:
                    try:
                        waveforms = np.array(nwbf.units['waveforms'][unit_idx][:])
                        unit_data['waveforms'] = waveforms
                    except Exception:
                        pass
                result['units'].append(unit_data)
    except Exception as e:
        print(f"Error processing {filepath.name}: {e}")
        import traceback
        traceback.print_exc()
    return result

def batch_extract_spike_data(filepaths, include_waveforms=False, quality_filters=None, include_trials=True, verbose=True):
    """
    Extract spike data for multiple NWB files.
    Args:
        filepaths: list of file paths or Path objects
        include_waveforms: bool
        quality_filters: dict
        include_trials: bool
        verbose: bool
    Returns:
        list of extract_spike_data results
    """
    if isinstance(filepaths, (str, Path)):
        filepaths = [filepaths]
    results = []
    n_files = len(filepaths)
    for i, filepath in enumerate(filepaths, 1):
        if verbose:
            print(f"Processing file {i}/{n_files}: {Path(filepath).name}")
        result = extract_spike_data(
            filepath, 
            include_waveforms=include_waveforms,
            quality_filters=quality_filters,
            include_trials=include_trials
        )
        results.append(result)
        if verbose:
            n_units = len(result['units'])
            n_trials = len(result['trials']) if result['trials'] is not None else 0
            print(f"  Extracted {n_units} units, {n_trials} trials")
    return results

def create_spike_dataframe(extraction_results):
    """
    Convert extraction results to DataFrame: one row per unit.
    Args:
        extraction_results: list of dict
    Returns:
        DataFrame
    """
    rows = []
    for result in extraction_results:
        for unit in result['units']:
            row = {
                'subject_id': result['subject_id'],
                'session_id': result['session_id'],
                'filepath': result['filepath'],
                'unit_id': unit['unit_id'],
                'n_spikes': unit['n_spikes'],
                'electrode_idx': unit['electrode_idx'],
                'brain_region': unit['brain_region'],
                'cluster_id': unit['cluster_id'],
                'mean_snr': unit['mean_snr'],
                'peak_snr': unit['peak_snr'],
                'isolation_distance': unit['isolation_distance'],
                'mean_proj_dist': unit['mean_proj_dist'],
                'spike_times': unit['spike_times']
            }
            rows.append(row)
    return pd.DataFrame(rows)

def extract_trials_dataframe(extraction_results):
    """
    Extract trial info from extraction results.
    Args:
        extraction_results: list of dict
    Returns:
        DataFrame with trial info, subject/session/filepath columns
    """
    trial_rows = []
    for result in extraction_results:
        if result['trials'] is not None:
            trials_df = result['trials'].copy()
            trials_df['subject_id'] = result['subject_id']
            trials_df['session_id'] = result['session_id']
            trials_df['filepath'] = result['filepath']
            trial_rows.append(trials_df)
    if len(trial_rows) == 0:
        return pd.DataFrame()
    return pd.concat(trial_rows, ignore_index=True)

def segment_spikes_by_trials(spike_times, trials_df, align_to='start_time', time_window=None):
    """
    Segment spikes by trials, aligned to align_to column.
    Args:
        spike_times: 1D array
        trials_df: DataFrame
        align_to: str, column to align to
        time_window: (start_offset, end_offset) or None
    Returns:
        list of arrays, one per trial (aligned)
    """
    segmented_spikes = []
    for trial_idx, trial in trials_df.iterrows():
        if align_to in trial and pd.notna(trial[align_to]) and trial[align_to] > 0:
            alignment_point = trial[align_to]
        elif 'start_time' in trial:
            alignment_point = trial['start_time']
        else:
            segmented_spikes.append(np.array([]))
            continue
        if time_window is not None:
            window_start = alignment_point + time_window[0]
            window_end = alignment_point + time_window[1]
        else:
            window_start = trial.get('start_time', alignment_point)
            window_end = trial.get('stop_time', alignment_point + 10.0)
        mask = (spike_times >= window_start) & (spike_times < window_end)
        trial_spikes = spike_times[mask]
        aligned_spikes = trial_spikes - alignment_point
        segmented_spikes.append(aligned_spikes)
    return segmented_spikes

def segment_spikes_by_selected_trials(spike_times, trials_df, selected_trial_indices=None,
                                      align_to='start_time', time_window=None):
    """
    Segment spikes for selected trials, as above but possibly a subset.
    Args:
        spike_times: 1D array
        trials_df: DataFrame
        selected_trial_indices: indices or None
        align_to: str
        time_window: tuple or None
    Returns:
        list of arrays, one per selected trial (aligned)
    """
    if selected_trial_indices is not None:
        selected_trials_df = trials_df.iloc[selected_trial_indices].copy()
        selected_trials_df = selected_trials_df.reset_index(drop=True)
    else:
        selected_trials_df = trials_df
    return segment_spikes_by_trials(
        spike_times,
        selected_trials_df,
        align_to=align_to,
        time_window=time_window
    )

def process_session(filepath, selected_trial_indices=None, align_to='start_time', time_window=None, 
                     trial_groups=None, group_labels=None):
    """
    Process a single session (file); returns DataFrame of units, with segmented spikes.
    Args:
        filepath: string or Path
        selected_trial_indices: indices, or None
        align_to: str
        time_window: tuple
        trial_groups: list of ndarrays or None (group indices within selected_trial_indices)
        group_labels: list of str or None
    Returns:
        session_df: DataFrame (rows = units)
    """
    result = extract_spike_data(filepath, include_trials=True)
    if result['trials'] is None or len(result['trials']) == 0:
        print(f"Warning: No trials found in {Path(filepath).name}")
        return None
    if len(result['units']) == 0:
        print(f"Warning: No units found in {Path(filepath).name}")
        return None
    trials_df = result['trials'].copy().reset_index(drop=True)
    if selected_trial_indices is None:
        selected_trial_indices = np.arange(len(trials_df))
    if trial_groups is not None and group_labels is None:
        group_labels = [f'group{i}' for i in range(len(trial_groups))]
    unit_rows = []
    for unit in result['units']:
        spike_times = unit['spike_times']
        segmented = segment_spikes_by_selected_trials(
            spike_times,
            trials_df,
            selected_trial_indices=selected_trial_indices,
            align_to=align_to,
            time_window=time_window
        )
        row = {
            'subject_id': result['subject_id'],
            'session_id': result['session_id'],
            'filepath': result['filepath'],
            'unit_id': unit['unit_id'],
            'n_spikes': unit['n_spikes'],
            'electrode_idx': unit['electrode_idx'],
            'brain_region': unit['brain_region'],
            'cluster_id': unit['cluster_id'],
            'mean_snr': unit['mean_snr'],
            'peak_snr': unit['peak_snr'],
            'isolation_distance': unit['isolation_distance'],
            'mean_proj_dist': unit['mean_proj_dist'],
            'spike_times': spike_times,
            'segmented_spikes': segmented,
            'n_trials': len(segmented),
            'selected_trial_indices': selected_trial_indices
        }
        if trial_groups is not None:
            row['trial_groups'] = trial_groups
            row['group_labels'] = group_labels
        unit_rows.append(row)
    session_df = pd.DataFrame(unit_rows)
    return session_df

def create_unit_metadata_df(extraction_result):
    """
    DataFrame of all metadata for each unit (no spikes).
    Args:
        extraction_result: result from extract_spike_data
    Returns:
        DataFrame
    """
    unit_rows = []
    for unit in extraction_result['units']:
        row = {
            'subject_id': extraction_result['subject_id'],
            'session_id': extraction_result['session_id'],
            'filepath': extraction_result['filepath'],
            'unit_id': unit['unit_id'],
            'n_spikes': unit['n_spikes'],
            'electrode_idx': unit['electrode_idx'],
            'brain_region': unit['brain_region'],
            'cluster_id': unit['cluster_id'],
            'mean_snr': unit['mean_snr'],
            'peak_snr': unit['peak_snr'],
            'isolation_distance': unit['isolation_distance'],
            'mean_proj_dist': unit['mean_proj_dist']
        }
        unit_rows.append(row)
    return pd.DataFrame(unit_rows)

def process_multiple_sessions(filepaths, trial_selection_func=None, align_to='start_time', time_window=None, 
                              trial_groups_dict=None, group_labels=None, verbose=True):
    """
    Process multiple sessions/filepaths; returns a compiled DataFrame.
    Args:
        filepaths: list of filepaths
        trial_selection_func: function (trials_df, filepath) -> indices, or None
        align_to, time_window: see process_session
        trial_groups_dict: dict mapping to group arrays
        group_labels: optional list of str for group_labels
        verbose: bool
    Returns:
        compiled_df: DataFrame
    """
    all_session_dfs = []
    for i, filepath in enumerate(filepaths, 1):
        if verbose:
            print(f"Processing session {i}/{len(filepaths)}: {Path(filepath).name}")
        result = extract_spike_data(filepath, include_trials=True, include_waveforms=False)
        if result['trials'] is None or len(result['trials']) == 0:
            if verbose:
                print(f"  Skipping: No trials found")
            continue
        trials_df = result['trials'].copy().reset_index(drop=True)
        if trial_selection_func is not None:
            selected_trial_indices = trial_selection_func(trials_df, filepath)
        else:
            selected_trial_indices = None
        if verbose and selected_trial_indices is not None:
            print(f"  Selected {len(selected_trial_indices)} trials")
        trial_groups = None
        if trial_groups_dict is not None:
            filepath_str = str(filepath)
            filepath_path = Path(filepath)
            if filepath_str in trial_groups_dict:
                trial_groups = trial_groups_dict[filepath_str]
            elif filepath_path in trial_groups_dict:
                trial_groups = trial_groups_dict[filepath_path]
            else:
                for key in trial_groups_dict:
                    if str(key) == filepath_str or str(key) == str(filepath_path):
                        trial_groups = trial_groups_dict[key]
                        break
                    if Path(key).name == filepath_path.name:
                        trial_groups = trial_groups_dict[key]
                        break
        session_df = process_session(
            filepath,
            selected_trial_indices=selected_trial_indices,
            align_to=align_to,
            time_window=time_window,
            trial_groups=trial_groups,
            group_labels=group_labels
        )
        if session_df is not None:
            all_session_dfs.append(session_df)
            if verbose:
                print(f"  Extracted {len(session_df)} units")
                if trial_groups is not None:
                    print(f"  Stored {len(trial_groups)} trial groups")
    if len(all_session_dfs) == 0:
        print("Warning: No sessions processed successfully")
        return pd.DataFrame()
    compiled_df = pd.concat(all_session_dfs, ignore_index=True)
    if verbose:
        print(f"\n✓ Compiled {len(compiled_df)} units from {len(all_session_dfs)} sessions")
        if trial_groups_dict is not None:
            print(f"  Group info stored in compiled_df columns")
    return compiled_df

def collapse_hemispheres(compiled_df, inplace=False):
    """
    Collapse left/right hemispheres for brain_region. Adds 'brain_region_collapsed' column.
    Args:
        compiled_df: DataFrame
        inplace: bool
    Returns:
        DataFrame with 'brain_region_collapsed'
    """
    if 'brain_region' not in compiled_df.columns:
        raise ValueError("DataFrame must have a 'brain_region' column")
    df = compiled_df if inplace else compiled_df.copy()
    def collapse_region(region):
        if pd.isna(region) or region is None:
            return region
        region_str = str(region).strip()
        if region_str.lower().endswith('_left'):
            return region_str[:-5]
        elif region_str.lower().endswith('_right'):
            return region_str[:-6]
        else:
            return region_str
    df['brain_region_collapsed'] = df['brain_region'].apply(collapse_region)
    return df

def list_available_brain_regions(compiled_df, show_collapsed=True):
    """
    Print all unique regions (and collapsed regions, if requested).
    Args:
        compiled_df: DataFrame with 'brain_region'
        show_collapsed: bool
    Returns:
        (original_regions, collapsed_regions)
    """
    if 'brain_region' not in compiled_df.columns:
        raise ValueError("DataFrame must have a 'brain_region' column")
    original_regions = compiled_df['brain_region'].dropna().unique()
    original_regions = sorted([str(r) for r in original_regions if r is not None])
    print("=== Available Brain Regions ===")
    print(f"\nOriginal regions ({len(original_regions)}):")
    for i, region in enumerate(original_regions, 1):
        count = len(compiled_df[compiled_df['brain_region'] == region])
        print(f"  {i:2d}. {region:30s} ({count} units)")
    if show_collapsed:
        if 'brain_region_collapsed' not in compiled_df.columns:
            df_temp = collapse_hemispheres(compiled_df.copy())
        else:
            df_temp = compiled_df
        collapsed_regions = df_temp['brain_region_collapsed'].dropna().unique()
        collapsed_regions = sorted([str(r) for r in collapsed_regions if r is not None])
        print(f"\nCollapsed regions ({len(collapsed_regions)}):")
        for i, region in enumerate(collapsed_regions, 1):
            count = len(df_temp[df_temp['brain_region_collapsed'] == region])
            print(f"  {i:2d}. {region:30s} ({count} units)")
        return original_regions, collapsed_regions
    return original_regions, None

def select_units_by_brain_region(compiled_df, brain_regions, match_type='exact', use_collapsed=True):
    """
    Select units by brain region.
    Args:
        compiled_df: DataFrame
        brain_regions: str or list
        match_type: 'exact' or 'contains'
        use_collapsed: bool
    Returns:
        filtered_df: DataFrame
    """
    if 'brain_region' not in compiled_df.columns:
        raise ValueError("DataFrame must have a 'brain_region' column")
    if use_collapsed:
        if 'brain_region_collapsed' not in compiled_df.columns:
            compiled_df = collapse_hemispheres(compiled_df.copy())
        region_col = 'brain_region_collapsed'
    else:
        region_col = 'brain_region'
    if isinstance(brain_regions, str):
        brain_regions = [brain_regions]
    if match_type == 'exact':
        mask = compiled_df[region_col].isin(brain_regions)
    elif match_type == 'contains':
        mask = compiled_df[region_col].astype(str).str.contains(
            '|'.join(brain_regions), case=False, na=False, regex=True
        )
    else:
        raise ValueError(f"Unknown match_type: {match_type}. Use 'exact' or 'contains'")
    filtered_df = compiled_df[mask].copy()
    return filtered_df

def assign_trial_conditions_per_unit(trials_df, encoding_epoch=1, 
                                     selected_trial_indices=None, n_conditions=5):
    """
    Assign each trial a 0-based condition index based on stimulus number (1..n_conditions).
    Ensures a fixed condition axis across sessions; missing/invalid IDs map to 0.
    Args:
        trials_df: DataFrame
        encoding_epoch: 1 or 3
        selected_trial_indices: indices or None
        n_conditions: total number of stimulus conditions (default 5)
    Returns:
        trial_conditions: np.array of length n_trials (0-based condition index)
    """
    pic_col = 'loadsEnc1_PicIDs' if encoding_epoch == 1 else 'loadsEnc3_PicIDs'
    trials_subset = trials_df.iloc[selected_trial_indices] if selected_trial_indices is not None else trials_df

    def map_pic_id(pic_id):
        if pd.isna(pic_id) or pic_id is None:
            return 0
        try:
            pid = int(pic_id)
        except Exception:
            return 0
        if pid <= 0:
            return 0
        idx = pid - 1
        # Clamp to configured condition count to avoid out-of-range indices
        return min(idx, n_conditions - 1)

    trial_conditions = np.array([map_pic_id(pic_id) for pic_id in trials_subset[pic_col]])
    return trial_conditions

def build_firing_rate_matrix_per_unit(segmented_spikes, trial_conditions, n_conditions,
                                      bin_size=0.02, time_range=None):
    """
    Build a unit's per-condition × timebin matrix (1 vector).
    Args:
        segmented_spikes: list of arrays, one per trial
        trial_conditions: np.array, int index per trial
        n_conditions: int
        bin_size: float, seconds
        time_range: (start, end) or None
    Returns:
        row_vector: 1D np.array (n_conditions * n_bins)
        bin_edges: np.array
    """
    if time_range is None:
        all_times = []
        for trial_spikes in segmented_spikes:
            if len(trial_spikes) > 0:
                all_times.extend(trial_spikes)
        if len(all_times) == 0:
            time_range = (0, 1)
        else:
            time_range = (min(all_times), max(all_times))
    start_time, end_time = time_range
    n_bins = int(np.ceil((end_time - start_time) / bin_size))
    bin_edges = np.linspace(start_time, end_time, n_bins + 1)
    row_vector = np.zeros(n_conditions * n_bins)
    for cond_idx in range(n_conditions):
        cond_mask = trial_conditions == cond_idx
        cond_trial_indices = np.where(cond_mask)[0]
        if len(cond_trial_indices) == 0:
            continue
        trial_binned = []
        for trial_idx in cond_trial_indices:
            trial_spikes = segmented_spikes[trial_idx]
            if len(trial_spikes) > 0:
                counts, _ = np.histogram(trial_spikes, bins=bin_edges)
                trial_binned.append(counts)
            else:
                trial_binned.append(np.zeros(n_bins))
        if len(trial_binned) > 0:
            cond_mean = np.mean(trial_binned, axis=0)
            start_col = cond_idx * n_bins
            end_col = (cond_idx + 1) * n_bins
            row_vector[start_col:end_col] = cond_mean
    return row_vector, bin_edges

def process_session_condition_based(filepath, selected_trial_indices=None, align_to='start_time', 
                                    time_window=None, encoding_epoch=1, encoding_time_window=(0.2, 1.0),
                                    bin_size=0.01, selected_unit_ids=None, n_conditions=5):
    """
    Process one session; returns condition-based firing rate matrix for units.
    Args:
        filepath: file
        selected_trial_indices: indices or None
        align_to: str (column in trials_df)
        time_window: tuple or None
        encoding_epoch: 1 or 3
        encoding_time_window: (start, end)
        bin_size: float
        selected_unit_ids: list of (subject_id, session_id, unit_id) or None
        n_conditions: total number of stimulus conditions (fixed across sessions)
    Returns:
        matrix: np.array (n_units, n_conditions * n_bins)
        unit_info: DataFrame
        bin_edges: np.array
        column_labels: list of str
        condition_info: dict
    """
    result = extract_spike_data(filepath, include_trials=True)
    if result['trials'] is None or len(result['trials']) == 0:
        return None, None, None, None, None
    trials_df = result['trials'].copy().reset_index(drop=True)
    if selected_trial_indices is None:
        selected_trial_indices = np.arange(len(trials_df))
    if encoding_epoch == 1:
        pic_col = 'loadsEnc1_PicIDs'
    else:
        pic_col = 'loadsEnc3_PicIDs'

    # Compute trial conditions once for all units (same for all units in a session)
    trial_conditions = assign_trial_conditions_per_unit(
        trials_df,
        encoding_epoch=encoding_epoch,
        selected_trial_indices=selected_trial_indices,
        n_conditions=n_conditions
    )

    selected_unit_set = set(selected_unit_ids) if selected_unit_ids is not None else None
    unit_rows = []
    matrix_rows = []
    bin_edges = None
    for unit in result['units']:
        unit_identifier = (result['subject_id'], result['session_id'], unit['unit_id'])
        if selected_unit_set is not None and unit_identifier not in selected_unit_set:
            continue
        spike_times = unit['spike_times']
        segmented = segment_spikes_by_selected_trials(
            spike_times, trials_df, selected_trial_indices,
            align_to=align_to, time_window=time_window
        )
        row_vector, bin_edges = build_firing_rate_matrix_per_unit(
            segmented, trial_conditions, n_conditions,
            bin_size=bin_size, time_range=time_window
        )
        matrix_rows.append(row_vector)
        unit_rows.append({
            'subject_id': result['subject_id'],
            'session_id': result['session_id'],
            'filepath': result['filepath'],
            'unit_id': unit['unit_id'],
            'n_spikes': unit['n_spikes'],
            'electrode_idx': unit['electrode_idx'],
            'brain_region': unit['brain_region'],
            'cluster_id': unit['cluster_id'],
            'mean_snr': unit['mean_snr'],
            'peak_snr': unit['peak_snr'],
            'isolation_distance': unit['isolation_distance'],
            'mean_proj_dist': unit['mean_proj_dist']
        })
    if len(matrix_rows) == 0:
        return None, None, None, None, None
    matrix = np.vstack(matrix_rows)
    unit_info = pd.DataFrame(unit_rows)
    n_bins = len(bin_edges) - 1 if bin_edges is not None else 0
    column_labels = []
    condition_info = {}
    for cond_idx in range(n_conditions):
        start_col = cond_idx * n_bins
        end_col = (cond_idx + 1) * n_bins
        condition_info[f'cond{cond_idx}'] = slice(start_col, end_col)
        for bin_idx in range(n_bins):
            column_labels.append(f'cond{cond_idx}_t{bin_idx}')
    return matrix, unit_info, bin_edges, column_labels, condition_info


## 1. load data

### all file list

In [ ]:
# List available files and select by index, with optional session filter
# session_filter options:
#   'all'   -> list all files
#   'ses-1' -> only session 1 files
#   'ses-2' -> only session 2 files
session_filter = 'ses-2'  # change as needed: 'all', 'ses-1', or 'ses-2'

# Get all files from disk, applying session_filter inside list_available_files
if session_filter == 'all':
    all_files = list_available_files(data_root)
else:
    all_files = list_available_files(data_root, session_filter=session_filter)


In [ ]:
# Select files to process from already-filtered all_files
# If selected_indices is empty [], all files in all_files will be selected
selected_indices = [0,1,2]  # e.g., [0, 1, 2]; leave [] to use all files

if len(selected_indices) == 0:
    selected_files = all_files
    print(f"Selected all {len(selected_files)} files to process:")
else:
    selected_files = [all_files[i] for i in selected_indices]
    print(f"Selected {len(selected_files)} files to process (manual indices):")

for f in selected_files:
    print(f"  {f.relative_to(data_root)}")

## 2. Unit and Trial Selection

For each session:
1. Extract unit metadata
2. Select units (e.g., by brain region)
3. Select trials (e.g., by correctness or load)
4. Build condition-based matrices (Section 3)

**Note:** Unit and trial selection happens per session, then matrices are compiled across sessions.


In [ ]:
# Define unit selection criteria (applied per session in Section 3)
# Select multiple brain regions to include in the compiled matrix
# Later in Section 5.5, filter into source and target subsets for PCA
selected_brain_regions = None  # Set to None to use all units (then filter in Section 5.5)
# Or specify a list of brain regions, e.g., ['hippocampus', 'pre_supplementary_motor_area']

# Define trial selection function: Correct trials AND load = 1
def select_correct_load1_trials(trials_df, filepath):
    """Select correct trials with load = 1 for this session."""
    # First select correct trials
    _, correct_indices = select_trials_by_correctness(trials_df, correct_only=True)
    
    # Then filter to load = 1
    correct_trials_df = trials_df.iloc[correct_indices]
    _, load1_indices_in_correct = select_trials_by_load(correct_trials_df, load_values=1)
    
    # Map back to original trial indices
    final_indices = correct_indices[load1_indices_in_correct]
    
    return final_indices


## 3. Build Firing Rate Matrices

Process sessions individually and build firing rate matrices with condition-based columns:
- **Projection epoch**: Aligned to maintenance onset, time window 0-2500 ms
- **Regression epoch**: Aligned to encoding 1 onset, time window 0-1000 ms
- Each row = one unit
- Each column = one time bin in one condition
- Conditions are based on stimulus numbers (1-5) from loadsEnc1_PicIDs/loadsEnc3_PicIDs
- To pool images from different subjects, we arbitrarily labeled the individual images of each subject as Images A–E. We then pooled the images with the same label across all subjects. Note that each subject saw a different set of pictures.
- Each value = mean spike count across trials in that condition for that unit in that bin
- Matrices are compiled vertically across sessions (all units have same number of conditions)


In [ ]:
# Process sessions individually and build firing rate matrices with condition-based columns
# Projection epoch: aligned to maintenance onset, time window 0-2500 ms
# Each column = one time bin in one condition (conditions based on stimulus numbers 1-5)
print("Processing projection epoch (0-2500 ms relative to maintenance onset) per session...")
print("Using condition-based matrix: each column = condition × time bin")

session_matrices_projection = []
session_unit_infos_projection = []
column_labels_list_projection = []
condition_info_list_projection = []
total_units_projection = 0

for i, filepath in enumerate(selected_files, 1):
    print(f"\nProcessing session {i}/{len(selected_files)}: {Path(filepath).name}")
    
    # Step 1: Extract unit metadata for this session
    result = extract_spike_data(filepath, include_trials=True)
    unit_metadata = create_unit_metadata_df(result)
    
    # Step 2: Select units for this session (e.g., by brain region)
    if selected_brain_regions is not None:
        selected_units = select_units_by_brain_region(
            unit_metadata, 
            brain_regions=selected_brain_regions,
            match_type='exact',
            use_collapsed=True
        )
        selected_unit_ids = [(row['subject_id'], row['session_id'], row['unit_id']) 
                             for _, row in selected_units.iterrows()]
        print(f"  Selected {len(selected_unit_ids)} units from {selected_brain_regions}")
    else:
        selected_unit_ids = None
        print(f"  Using all {len(unit_metadata)} units")
    
    # Step 3: Select trials for this session (e.g., by correctness)
    trials_df = result['trials']
    if trials_df is None or len(trials_df) == 0:
        print(f"  Skipping: No trials found")
        continue
    
    selected_trial_indices = select_correct_load1_trials(trials_df, filepath)
    print(f"  Selected {len(selected_trial_indices)} trials (correct + load=1)")
    
    # Step 4: Build matrices for this session (selected units + selected trials)
    matrix, unit_info, bin_edges, column_labels, condition_info = process_session_condition_based(
        filepath,
        selected_unit_ids=selected_unit_ids,
        selected_trial_indices=selected_trial_indices,
        align_to='timestamps_Maintenance',
        time_window=(0.0, 2.5),
        encoding_epoch=1,
        encoding_time_window=(0.2, 1.0),
        bin_size=0.01
    )
    
    if matrix is None or len(matrix) == 0:
        print(f"  Skipping: No units processed")
        continue
    
    # Step 5: Store session matrix
    if matrix is not None and len(matrix) > 0:
        session_matrices_projection.append(matrix)
        session_unit_infos_projection.append(unit_info)
        column_labels_list_projection.append(column_labels)
        condition_info_list_projection.append(condition_info)
        total_units_projection += len(matrix)
        
        print(f"  Built matrix: {len(matrix)} units, shape {matrix.shape}")
        if condition_info is not None:
            print(f"    Conditions: {len(condition_info)}, Bins per condition: {len(bin_edges) - 1}")

# Step 6: Compile all session matrices vertically
if len(session_matrices_projection) > 0:
    # Verify all matrices have same number of columns
    n_cols = session_matrices_projection[0].shape[1]
    for i, matrix in enumerate(session_matrices_projection[1:], 1):
        if matrix.shape[1] != n_cols:
            print(f"Warning: Session {i} has {matrix.shape[1]} columns, expected {n_cols}")
    
    # Stack matrices vertically
    firing_rate_projection = np.vstack(session_matrices_projection)
    unit_info_projection = pd.concat(session_unit_infos_projection, ignore_index=True)
    
    # Use first session's column labels and condition info (should be consistent)
    column_labels_projection = column_labels_list_projection[0] if len(column_labels_list_projection) > 0 else None
    condition_info_projection = condition_info_list_projection[0] if len(condition_info_list_projection) > 0 else None
    bin_edges_projection = bin_edges
    
    print(f"\n=== Projection Epoch Results ===")
    print(f"Total units: {total_units_projection}")
    print(f"Matrix shape: {firing_rate_projection.shape} (units × conditions × time bins)")
    if condition_info_projection is not None:
        n_conditions = len(condition_info_projection)
        n_bins_per_condition = len(bin_edges_projection) - 1
        print(f"  Number of conditions: {n_conditions}")
        print(f"  Number of time bins per condition: {n_bins_per_condition}")
        print(f"  Total columns: {n_conditions} × {n_bins_per_condition} = {n_conditions * n_bins_per_condition}")
    print(f"Bin size: 0.01s (10ms)")
    print(f"Time range: {bin_edges_projection[0]:.3f}s to {bin_edges_projection[-1]:.3f}s")
    print(f"✓ Compiled {len(session_matrices_projection)} sessions")
else:
    print("No units found. Check file selection and trial data.")
    firing_rate_projection = None
    unit_info_projection = None
    column_labels_projection = None
    condition_info_projection = None
    bin_edges_projection = None


In [ ]:
# Process sessions individually and build firing rate matrices with condition-based columns
# Regression epoch: aligned to encoding 1 onset, time window 0-1000 ms
# Each column = one time bin in one condition (conditions based on stimulus numbers 1-5)
print("Processing regression epoch (0-1000 ms relative to encoding 1 onset) per session...")
print("Using condition-based matrix: each column = condition × time bin")

session_matrices_regression = []
session_unit_infos_regression = []
column_labels_list_regression = []
condition_info_list_regression = []
total_units_regression = 0

for i, filepath in enumerate(selected_files, 1):
    print(f"\nProcessing session {i}/{len(selected_files)}: {Path(filepath).name}")
    
    # Step 1: Extract unit metadata for this session
    result = extract_spike_data(filepath, include_trials=True)
    unit_metadata = create_unit_metadata_df(result)
    
    # Step 2: Select units for this session (e.g., by brain region)
    if selected_brain_regions is not None:
        selected_units = select_units_by_brain_region(
            unit_metadata, 
            brain_regions=selected_brain_regions,
            match_type='exact',
            use_collapsed=True
        )
        selected_unit_ids = [(row['subject_id'], row['session_id'], row['unit_id']) 
                             for _, row in selected_units.iterrows()]
        print(f"  Selected {len(selected_unit_ids)} units from {selected_brain_regions}")
    else:
        selected_unit_ids = None
        print(f"  Using all {len(unit_metadata)} units")
    
    # Step 3: Select trials for this session (e.g., by correctness)
    trials_df = result['trials']
    if trials_df is None or len(trials_df) == 0:
        print(f"  Skipping: No trials found")
        continue
    
    selected_trial_indices = select_correct_load1_trials(trials_df, filepath)
    print(f"  Selected {len(selected_trial_indices)} trials (correct + load=1)")
    
    # Step 4: Build matrices for this session (selected units + selected trials)
    matrix, unit_info, bin_edges, column_labels, condition_info = process_session_condition_based(
        filepath,
        selected_unit_ids=selected_unit_ids,
        selected_trial_indices=selected_trial_indices,
        align_to='timestamps_Encoding1',
        time_window=(0.0, 1.0),
        encoding_epoch=1,
        encoding_time_window=(0.2, 1.0),
        bin_size=0.01
    )
    
    if matrix is None or len(matrix) == 0:
        print(f"  Skipping: No units processed")
        continue
    
    # Step 5: Store session matrix
    if matrix is not None and len(matrix) > 0:
        session_matrices_regression.append(matrix)
        session_unit_infos_regression.append(unit_info)
        column_labels_list_regression.append(column_labels)
        condition_info_list_regression.append(condition_info)
        total_units_regression += len(matrix)
        
        print(f"  Built matrix: {len(matrix)} units, shape {matrix.shape}")
        if condition_info is not None:
            print(f"    Conditions: {len(condition_info)}, Bins per condition: {len(bin_edges) - 1}")

# Step 6: Compile all session matrices vertically
if len(session_matrices_regression) > 0:
    # Verify all matrices have same number of columns
    n_cols = session_matrices_regression[0].shape[1]
    for i, matrix in enumerate(session_matrices_regression[1:], 1):
        if matrix.shape[1] != n_cols:
            print(f"Warning: Session {i} has {matrix.shape[1]} columns, expected {n_cols}")
    
    # Stack matrices vertically
    firing_rate_regression = np.vstack(session_matrices_regression)
    unit_info_regression = pd.concat(session_unit_infos_regression, ignore_index=True)
    
    # Use first session's column labels and condition info (should be consistent)
    column_labels_regression = column_labels_list_regression[0] if len(column_labels_list_regression) > 0 else None
    condition_info_regression = condition_info_list_regression[0] if len(condition_info_list_regression) > 0 else None
    bin_edges_regression = bin_edges
    
    print(f"\n=== Regression Epoch Results ===")
    print(f"Total units: {total_units_regression}")
    print(f"Matrix shape: {firing_rate_regression.shape} (units × conditions × time bins)")
    if condition_info_regression is not None:
        n_conditions = len(condition_info_regression)
        n_bins_per_condition = len(bin_edges_regression) - 1
        print(f"  Number of conditions: {n_conditions}")
        print(f"  Number of time bins per condition: {n_bins_per_condition}")
        print(f"  Total columns: {n_conditions} × {n_bins_per_condition} = {n_conditions * n_bins_per_condition}")
    print(f"Bin size: 0.01s (10ms)")
    print(f"Time range: {bin_edges_regression[0]:.3f}s to {bin_edges_regression[-1]:.3f}s")
    print(f"✓ Compiled {len(session_matrices_regression)} sessions")
else:
    print("No units found. Check file selection and trial data.")
    firing_rate_regression = None
    unit_info_regression = None
    column_labels_regression = None
    condition_info_regression = None
    bin_edges_regression = None


In [ ]:
# Summary comparison of both matrices
print("=== Comparison Summary ===")
print(f"\nProjection epoch matrix:")
if firing_rate_projection is not None:
    print(f"  Units: {firing_rate_projection.shape[0]}")
    print(f"  Time bins: {firing_rate_projection.shape[1]}")
    print(f"  Mean firing rate: {firing_rate_projection.mean():.2f} spikes/bin")
else:
    print(f"  No data")
print(f"  Time window: 0-2500 ms relative to maintenance onset")

print(f"\nRegression epoch matrix:")
if firing_rate_regression is not None:
    print(f"  Units: {firing_rate_regression.shape[0]}")
    print(f"  Time bins: {firing_rate_regression.shape[1]}")
    print(f"  Mean firing rate: {firing_rate_regression.mean():.2f} spikes/bin")
else:
    print(f"  No data")
print(f"  Time window: 0-1000 ms relative to encoding 1 onset")

print(f"\n✓ Both matrices ready for null space analysis")


## 4. Combine Projection and Regression Epoch Matrices

Combine projection epoch and regression epoch matrices for the same units by concatenating time bins **horizontally**:
- Matches common units between the two matrices
- Concatenates projection epoch columns (0-2500 ms relative to maintenance onset) and regression epoch columns (0-1000 ms relative to encoding 1 onset) side-by-side
- Creates the **source matrix** for null space analysis
- Note: This is horizontal concatenation (different from vertical compilation in Section 3)


In [ ]:
def combine_firing_rate_matrices(firing_rate_projection, unit_info_projection,
                                  firing_rate_regression, unit_info_regression,
                                  column_labels_projection=None, group_info_projection=None,
                                  column_labels_regression=None, group_info_regression=None):
    """
    Combine two firing rate matrices for the same units by concatenating horizontally.
    Handles group×time_bin column structure.
    
    Args:
        firing_rate_projection: Array of shape (n_units_projection, n_bins_projection) or (n_units_projection, n_groups × n_bins_projection)
        unit_info_projection: DataFrame with unit metadata for projection epoch matrix
        firing_rate_regression: Array of shape (n_units_regression, n_bins_regression) or (n_units_regression, n_groups × n_bins_regression)
        unit_info_regression: DataFrame with unit metadata for regression epoch matrix
        column_labels_projection: List of column labels for projection epoch matrix (if grouped), or None
        group_info_projection: Dict mapping group names to column index ranges for projection epoch (if grouped), or None
        column_labels_regression: List of column labels for regression epoch matrix (if grouped), or None
        group_info_regression: Dict mapping group names to column index ranges for regression epoch (if grouped), or None
    
    Returns:
        combined_matrix: Array of shape (n_common_units, n_bins_projection + n_bins_regression) or (n_common_units, n_groups × (n_bins_projection + n_bins_regression))
        combined_unit_info: DataFrame with metadata for common units
        projection_indices: Indices of common units in projection epoch matrix
        regression_indices: Indices of common units in regression epoch matrix
        combined_column_labels: List of combined column names, or None
        combined_group_info: Updated group info dict for combined matrix, or None
    """
    # Validate inputs
    if unit_info_projection is None:
        raise ValueError("unit_info_projection is None. No units were found in the projection epoch data. "
                        "Please check your data loading and filtering criteria.")
    if unit_info_regression is None:
        raise ValueError("unit_info_regression is None. No units were found in the regression epoch data. "
                        "Please check your data loading and filtering criteria.")
    if firing_rate_projection is None:
        raise ValueError("firing_rate_projection is None. No firing rate data was found for the projection epoch.")
    if firing_rate_regression is None:
        raise ValueError("firing_rate_regression is None. No firing rate data was found for the regression epoch.")
    
    # Create unique identifiers for units: (subject_id, session_id, unit_id)
    def create_unit_id(df):
        return df.apply(lambda row: (row['subject_id'], row['session_id'], row['unit_id']), axis=1)
    
    unit_ids_projection = create_unit_id(unit_info_projection)
    unit_ids_regression = create_unit_id(unit_info_regression)
    
    # Find common units
    common_unit_ids = set(unit_ids_projection) & set(unit_ids_regression)
    
    if len(common_unit_ids) == 0:
        raise ValueError("No common units found between the two matrices")
    
    # Get indices for common units in both matrices
    projection_indices = []
    regression_indices = []
    
    for unit_id in sorted(common_unit_ids):
        proj_idx = unit_ids_projection[unit_ids_projection == unit_id].index[0]
        reg_idx = unit_ids_regression[unit_ids_regression == unit_id].index[0]
        projection_indices.append(proj_idx)
        regression_indices.append(reg_idx)
    
    # Extract rows for common units
    firing_rate_projection_common = firing_rate_projection[projection_indices, :]
    firing_rate_regression_common = firing_rate_regression[regression_indices, :]
    
    # Concatenate horizontally: (n_common_units, n_bins_projection + n_bins_regression)
    combined_matrix = np.hstack([firing_rate_projection_common, firing_rate_regression_common])
    
    # Extract unit info for common units (use projection as reference)
    combined_unit_info = unit_info_projection.iloc[projection_indices].copy().reset_index(drop=True)
    
    # Combine column labels and group info if provided
    combined_column_labels = None
    combined_group_info = None
    
    if column_labels_projection is not None or column_labels_regression is not None:
        combined_column_labels = []
        
        # Add projection epoch prefix to projection columns
        if column_labels_projection is not None:
            for label in column_labels_projection:
                combined_column_labels.append(f'projection_{label}')
        else:
            # No labels, create default
            n_cols_proj = firing_rate_projection_common.shape[1]
            combined_column_labels.extend([f'projection_t{i}' for i in range(n_cols_proj)])
        
        # Add regression epoch prefix to regression columns
        if column_labels_regression is not None:
            for label in column_labels_regression:
                combined_column_labels.append(f'regression_{label}')
        else:
            # No labels, create default
            n_cols_reg = firing_rate_regression_common.shape[1]
            combined_column_labels.extend([f'regression_t{i}' for i in range(n_cols_reg)])
        
        # Update group_info if provided
        if group_info_projection is not None and group_info_regression is not None:
            combined_group_info = {}
            n_cols_proj = firing_rate_projection_common.shape[1]
            
            # Update projection group slices
            for group_name, group_slice in group_info_projection.items():
                start = group_slice.start
                end = group_slice.stop
                combined_group_info[f'projection_{group_name}'] = slice(start, end)
            
            # Update regression group slices (offset by projection columns)
            for group_name, group_slice in group_info_regression.items():
                start = group_slice.start + n_cols_proj
                end = group_slice.stop + n_cols_proj
                combined_group_info[f'regression_{group_name}'] = slice(start, end)
    
    return combined_matrix, combined_unit_info, np.array(projection_indices), np.array(regression_indices), combined_column_labels, combined_group_info


In [ ]:
# Combine the two matrices (Step 8)
print("Combining projection and regression epoch matrices...")
firing_rate_combined, unit_info_combined, projection_indices, regression_indices, combined_column_labels, combined_group_info = combine_firing_rate_matrices(
    firing_rate_projection,
    unit_info_projection,
    firing_rate_regression,
    unit_info_regression,
    column_labels_projection=column_labels_projection,
    group_info_projection=condition_info_projection,  # Using condition_info instead of group_info
    column_labels_regression=column_labels_regression,
    group_info_regression=condition_info_regression   # Using condition_info instead of group_info
)

print(f"\n=== Combined Firing Rate Matrix ===")
print(f"Matrix shape: {firing_rate_combined.shape} (units × conditions × time bins)")
print(f"  - Units: {firing_rate_combined.shape[0]} (common units only)")
print(f"  - Total columns: {firing_rate_combined.shape[1]}")
if condition_info_projection is not None:
    n_conditions = len(condition_info_projection)
    n_bins_projection = firing_rate_projection.shape[1] // n_conditions
    n_bins_regression = firing_rate_regression.shape[1] // n_conditions
    print(f"    * Projection epoch: {n_conditions} conditions × {n_bins_projection} bins = {firing_rate_projection.shape[1]} columns (0-2000 ms)")
    print(f"    * Regression epoch: {n_conditions} conditions × {n_bins_regression} bins = {firing_rate_regression.shape[1]} columns (0-1000 ms relative to encoding 1 onset)")
else:
    print(f"    * Projection epoch bins: {firing_rate_projection.shape[1]} (0-2000 ms)")
    print(f"    * Regression epoch bins: {firing_rate_regression.shape[1]} (0-1000 ms relative to encoding 1 onset)")
print(f"    * Total: {firing_rate_projection.shape[1] + firing_rate_regression.shape[1]} columns")
print(f"\nMean firing rate: {firing_rate_combined.mean():.2f} spikes/bin")
print(f"Max firing rate: {firing_rate_combined.max():.2f} spikes/bin")
print(f"\n✓ Combined matrix ready for null space analysis")


In [ ]:
# Verify unit matching
print("=== Unit Matching Verification ===")
print(f"Units in projection epoch matrix: {len(unit_info_projection)}")
print(f"Units in regression epoch matrix: {len(unit_info_regression)}")
print(f"Common units: {len(unit_info_combined)}")
print(f"\nFirst few common units:")
print(unit_info_combined[['subject_id', 'session_id', 'unit_id']].head())


In [ ]:
# Create index vectors to track which columns belong to which epoch (Step 9)
# Updated to work with condition×time_bin column structure
n_cols_projection = firing_rate_projection.shape[1]
n_cols_regression = firing_rate_regression.shape[1]
n_cols_total = firing_rate_combined.shape[1]

# idx_regression: columns belonging to regression epoch - all conditions
idx_regression = np.zeros(n_cols_total, dtype=bool)
idx_regression[n_cols_projection:] = True  # All regression epoch columns (across all conditions)

# idx_project: columns belonging to projection epoch - all conditions
idx_project = np.zeros(n_cols_total, dtype=bool)
idx_project[:n_cols_projection] = True  # All projection epoch columns (across all conditions)

print("=== Column Index Vectors ===")
print(f"Total columns in combined matrix: {n_cols_total}")
print(f"  Projection epoch (idx_project): {idx_project.sum()} columns (indices 0-{n_cols_projection-1})")
print(f"  Regression epoch (idx_regression): {idx_regression.sum()} columns (indices {n_cols_projection}-{n_cols_total-1})")
print(f"\nVerification:")
print(f"  idx_project + idx_regression should equal all True: {(idx_project | idx_regression).all()}")
print(f"  No overlap: {(idx_project & idx_regression).sum() == 0}")

# Optionally create condition-specific indices if condition info is available
if combined_group_info is not None:
    print(f"\n=== Condition-Specific Indices ===")
    condition_indices = {}
    for condition_name, condition_slice in combined_group_info.items():
        condition_mask = np.zeros(n_cols_total, dtype=bool)
        condition_mask[condition_slice] = True
        condition_indices[condition_name] = condition_mask
        
        # Check if this condition belongs to projection or regression epoch
        epoch_type = 'projection' if 'projection' in condition_name else 'regression'
        print(f"  {condition_name}: {condition_mask.sum()} columns ({epoch_type} epoch)")
    
    print(f"\n  Example: idx_regression_cond0 = condition_indices['regression_cond0']")
    print(f"           idx_project_cond0 = condition_indices['projection_cond0']")
elif condition_info_projection is not None:
    # Use condition_info from projection epoch (should be same structure)
    print(f"\n=== Condition Structure ===")
    n_conditions = len(condition_info_projection)
    n_bins_projection = len(bin_edges_projection) - 1
    n_bins_regression = len(bin_edges_regression) - 1
    print(f"  Number of conditions: {n_conditions}")
    print(f"  Bins per condition (projection): {n_bins_projection}")
    print(f"  Bins per condition (regression): {n_bins_regression}")
    print(f"  Total projection columns: {n_conditions} × {n_bins_projection} = {n_cols_projection}")
    print(f"  Total regression columns: {n_conditions} × {n_bins_regression} = {n_cols_regression}")

print(f"\n✓ Index vectors created:")
print(f"  idx_regression: regression epoch columns - all conditions")
print(f"  idx_project: projection epoch columns (project/prep) - all conditions")


## 5. Normalize Matrix

Normalize each neuron (row) by:
1. Mean-centering: subtract row mean
2. Range-normalization: divide by row range (max - min)


In [ ]:
def normalize_matrix_range_mean(matrix):
    """
    Normalize matrix by mean-centering and range-normalization for each row (neuron).
    
    For each row:
    1. Subtract row mean (mean-center)
    2. Divide by row range (max - min) (range-normalize)
    
    Args:
        matrix: Array of shape (n_neurons, n_features)
    
    Returns:
        normalized_matrix: Array of same shape, normalized per row
        row_means: Array of row means (for potential denormalization)
        row_ranges: Array of row ranges (for potential denormalization)
    """
    matrix = np.array(matrix, dtype=float)
    n_neurons, n_features = matrix.shape
    
    # Compute row means
    row_means = np.mean(matrix, axis=1, keepdims=True)  # Shape: (n_neurons, 1)
    
    # Mean-center: subtract row mean
    matrix_centered = matrix - row_means
    
    # Compute row ranges (max - min)
    row_max = np.max(matrix, axis=1, keepdims=True)  # Shape: (n_neurons, 1)
    row_min = np.min(matrix, axis=1, keepdims=True)  # Shape: (n_neurons, 1)
    row_ranges = row_max - row_min  # Shape: (n_neurons, 1)
    
    # Handle edge case: if range is 0 (constant row), set to 1 to avoid division by zero
    row_ranges[row_ranges == 0] = 1.0
    
    # Range-normalize: divide by row range
    normalized_matrix = matrix_centered / row_ranges
    
    return normalized_matrix, row_means.squeeze(), row_ranges.squeeze()


In [ ]:
# Normalize the combined firing rate matrix
print("Normalizing combined firing rate matrix...")
firing_rate_normalized, row_means, row_ranges = normalize_matrix_range_mean(firing_rate_combined)

print(f"\n=== Normalization Results ===")
print(f"Original matrix:")
print(f"  Mean across all values: {firing_rate_combined.mean():.4f}")
print(f"  Std across all values: {firing_rate_combined.std():.4f}")
print(f"  Min: {firing_rate_combined.min():.4f}, Max: {firing_rate_combined.max():.4f}")

print(f"\nNormalized matrix:")
print(f"  Mean across all values: {firing_rate_normalized.mean():.4f} (should be ~0)")
print(f"  Std across all values: {firing_rate_normalized.std():.4f}")
print(f"  Min: {firing_rate_normalized.min():.4f}, Max: {firing_rate_normalized.max():.4f}")

print(f"\nPer-row statistics:")
print(f"  Row means (original): mean={row_means.mean():.4f}, std={row_means.std():.4f}")
print(f"  Row ranges (original): mean={row_ranges.mean():.4f}, std={row_ranges.std():.4f}")
print(f"  Number of constant rows (range=0): {(row_ranges == 0).sum()}")

print(f"\n✓ Matrix normalized (mean-centered + range-normalized per row)")


## 5.5. Brain Region Selection (Source & Target)

Filter the compiled matrix into **source** and **target** subsets by brain region for separate dPCA analysis.

- **Source regions**: Used for Source dPCA (Section 6) - e.g., hippocampus
- **Target regions**: Used for Target dPCA (Section 7) - e.g., pre_supplementary_motor_area

Note: The compiled matrix already contains only units from `selected_brain_regions` (Section 2).


In [ ]:
# Step 1: List available brain regions
if len(unit_info_combined) > 0 and 'brain_region' in unit_info_combined.columns:
    original_regions, collapsed_regions = list_available_brain_regions(
        unit_info_combined, show_collapsed=True
    )
else:
    print("No brain region data available in unit_info_combined")


In [ ]:
# Define source and target brain regions for PCA
# NOTE: Original matrix and unit_info remain untouched
source_regions = ['hippocampus']
target_regions = ['pre_supplementary_motor_area']

# --- Source subset ---
unit_info_source_subset = select_units_by_brain_region(
    unit_info_combined,
    brain_regions=source_regions,
    match_type='exact',
    use_collapsed=True
)
source_indices = unit_info_source_subset.index.values
firing_rate_normalized_source = firing_rate_normalized[source_indices, :]
unit_info_combined_source = unit_info_source_subset.reset_index(drop=True)

# --- Target subset ---
unit_info_target_subset = select_units_by_brain_region(
    unit_info_combined,
    brain_regions=target_regions,
    match_type='exact',
    use_collapsed=True
)
target_indices = unit_info_target_subset.index.values
firing_rate_normalized_target = firing_rate_normalized[target_indices, :]
unit_info_combined_target = unit_info_target_subset.reset_index(drop=True)

print(f"=== Brain Region Subsets Created ===")
print(f"Total units in compiled matrix: {len(unit_info_combined)}")
print(f"\nSource: {source_regions}")
print(f"  Units: {len(unit_info_combined_source)}")
print(f"  Matrix shape: {firing_rate_normalized_source.shape}")
print(f"\nTarget: {target_regions}")
print(f"  Units: {len(unit_info_combined_target)}")
print(f"  Matrix shape: {firing_rate_normalized_target.shape}")
print(f"\n✓ Source and target subsets ready for PCA")

In [ ]:
# PCA Implementation Functions (standard SVD-based)

import numpy as np

def compute_pca(data_2d, n_components=None):
    """
    Standard PCA on rows = samples, cols = features.
    Centers features and returns components in feature space.
    """
    data = np.asarray(data_2d, dtype=float)
    n_samples, n_features = data.shape
    mean = data.mean(axis=0, keepdims=True)
    X = data - mean

    # SVD-based PCA
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    max_rank = min(n_samples, n_features)
    if n_components is None:
        n_components = max_rank
    n_components = min(n_components, max_rank)

    components = Vt[:n_components]
    explained_variance = (S[:n_components] ** 2) / max(1, n_samples - 1)
    total_variance = (S ** 2).sum() / max(1, n_samples - 1)
    explained_variance_ratio = explained_variance / total_variance if total_variance > 0 else np.zeros_like(explained_variance)

    return {
        'components': components,
        'explained_variance': explained_variance,
        'explained_variance_ratio': explained_variance_ratio,
        'total_variance': total_variance,
        'mean': mean.ravel(),
        'n_samples': n_samples,
        'n_features': n_features
    }


def transform_pca(data_2d, pca_result):
    """
    Project data onto PCA components using stored mean.
    """
    mean = pca_result.get('mean', None)
    if mean is None:
        mean = np.mean(data_2d, axis=0)
    X = np.asarray(data_2d, dtype=float) - mean
    components = pca_result['components']
    return X @ components.T


print("✓ PCA functions defined (standard SVD-based)")


## 6. PCA Analysis (Source)

Run PCA on the source subset matrix (`firing_rate_normalized_source`) to find principal components across time bins (conditions already stacked in the feature dimension).

**Naming Convention:**
- `firing_rate_normalized_source`: Subset matrix containing only units from selected source brain regions
- `firing_rate_normalized`: Full matrix with all units (NOT used in this section)
- `k_source`: Number of PCA components selected for source (>50% variance threshold)

**PCA Configuration:**
- Rows = units, columns = condition × time-bin features
- Components capture maximal variance (no demixing)


In [ ]:

# Run PCA on source matrix (time × neurons)
# Transpose so PCA compresses neurons and yields low-dim latent activity over time
print("Running PCA on source matrix (time × neurons)...")

# Build time-by-neuron matrix
X_source = firing_rate_normalized_source.T  # shape: (total_time_bins, n_neurons_source)
T_all_source, n_neurons_source = X_source.shape

print(f"Matrix shape: {X_source.shape} (time bins × neurons)")
print(f"  Time bins (all epochs): {T_all_source}")
print(f"  Neurons: {n_neurons_source}")

print("=== Running PCA on full matrix (variance profile) ===")
pca_source_full = compute_pca(X_source, n_components=None)
explained_variance_ratio = pca_source_full['explained_variance_ratio']
cumulative_variance = np.cumsum(explained_variance_ratio)
n_components = len(explained_variance_ratio)

print("=== PCA Results (Full Matrix) ===")
print(f"Total components: {n_components}")
print(f"Total explained variance: {cumulative_variance[-1]:.4f}")
print("First 10 components:")
for i in range(min(10, n_components)):
    print(f"  PC{i+1}: {explained_variance_ratio[i]:.4f} ({explained_variance_ratio[i]*100:.2f}%)")


In [ ]:

# Find k where cumulative explained variance passes threshold (>50%)
variance_threshold = 0.5  # >50% variance threshold

# Find first component where cumulative variance > threshold
elbow_idx = np.where(cumulative_variance > variance_threshold)[0]
if len(elbow_idx) > 0:
    elbow_k = elbow_idx[0] + 1  # Convert to 1-indexed component number
else:
    # If threshold not reached, use all components
    elbow_k = len(cumulative_variance)

print("=== Component Selection (Threshold Method) ===")
print(f"Variance threshold: >{variance_threshold*100:.0f}%")
print(f"Selected k: {elbow_k} components")
print(f"Cumulative variance at k={elbow_k}: {cumulative_variance[elbow_k-1]:.4f} ({cumulative_variance[elbow_k-1]*100:.2f}%)")


In [ ]:

# Visualize cumulative explained variance and threshold in a clean format
fig, ax = plt.subplots(figsize=(10, 6))

# Plot cumulative explained variance
ax.plot(
    range(1, min(101, n_components + 1)),
    cumulative_variance[:min(100, n_components)],
    marker='o', markersize=3, linewidth=2,
    label='Cumulative explained variance', color='steelblue'
)

# Mark selected k with vertical line
ax.axvline(
    x=elbow_k, color='red', linestyle='--', linewidth=2.5,
    label=f'Selected k={elbow_k}'
)
# Mark threshold line
ax.axhline(
    y=variance_threshold, color='orange', linestyle='--',
    linewidth=2, label=f'Threshold = >{int(variance_threshold*100)}%'
)

# Add annotation at intersection
annotation_text = f'k={elbow_k}({cumulative_variance[elbow_k-1]*100:.1f}%)'
ax.text(
    elbow_k, variance_threshold + 0.02,
    annotation_text,
    fontsize=10, ha='center',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
)

# Formatting axes and plot
ax.set_xlabel('Number of Components (k)', fontsize=12)
ax.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax.set_title('Cumulative Explained Variance - PCA Component Selection (All Epochs)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, min(101, n_components + 1))
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f"✓ Cumulative variance plot: k={elbow_k} selected at >{int(variance_threshold*100)}% threshold")
print(f"  To adjust: change 'variance_threshold' variable (currently {variance_threshold})")


In [ ]:

# Extract PCA components with k_source components (time × neuron PCA)
k_source = int(elbow_k)  # Rename for clarity - ensure int
max_k_source = min(T_all_source, n_neurons_source)

# Cap k_source at max allowed
if k_source > max_k_source:
    print(f"Warning: k_source={k_source} exceeds max allowed ({max_k_source}). Capping to {max_k_source}.")
    k_source = max_k_source

print("=== Extracting PCA Components ===")
print(f"Selected k_source: {k_source} components")
print(f"  Matrix shape: {X_source.shape} (time bins={T_all_source}, neurons={n_neurons_source})")
print(f"  Max k allowed: {max_k_source}")

# Run PCA on full matrix with k_source components
pca_source_k = compute_pca(X_source, n_components=k_source)

# Components: neuron weights (k_source × n_neurons)
principal_components_source = pca_source_k['components']

# Scores: latent activity over time (time bins × k_source)
scores_source = transform_pca(X_source, pca_source_k)

# Use latent time series (k_source × time bins) for downstream regression/null-space
scores_source_full = scores_source.T

print("=== Source PCA Components & Scores ===")
print(f"Principal components (neuron weights) shape: {principal_components_source.shape} (k_source × n_neurons)")
print(f"Scores shape: {scores_source.shape} (time bins × k_source)")
print(f"Latent time series shape: {scores_source_full.shape} (k_source × time bins)")

print("Explained variance (k_source components):")
total_var = pca_source_k['explained_variance_ratio'].sum()
print(f"  Total: {total_var:.4f} ({total_var*100:.2f}%)")
print(f"  Per component:")
for i in range(min(k_source, 10)):
    ev = pca_source_k['explained_variance_ratio'][i]
    print(f"    PC{i+1}: {ev:.4f} ({ev*100:.2f}%)")

# Store for later steps
pca_model_source = pca_source_k           # PCA model dictionary (time × neuron)
k_components_source = k_source
pc_matrix_source = scores_source_full     # k_source × time bins (used downstream)
score_matrix_source = scores_source       # time bins × k_source


In [ ]:

# Store and summarize source PCA results (formatted output)
print("=== Stored Source PCA Results ===")
print(f"  k_source: {k_components_source}")
print(f"  Latent time series (pc_matrix_source) shape: {pc_matrix_source.shape}")
print(f"  Scores matrix shape (time × k): {score_matrix_source.shape}")
print(f"  Neuron-weight matrix shape: {pca_model_source['components'].shape}")
print(f"  Explained variance ratio: {pca_model_source['explained_variance_ratio']}")
print(f"  Cumulative variance (up to k_source): {np.cumsum(pca_model_source['explained_variance_ratio'])[-1]:.4f}")

print("✓ Source PCA results stored and ready.")
print(f"  - Components compress neurons; pc_matrix_source provides latent activity over time")
print(f"  - Next: run target PCA (Section 7) with k_target = k_source // 2")


## 7. PCA Analysis (Target)

Run PCA on the target subset matrix (`firing_rate_normalized_target`) using k_target = k_source / 2.

**Naming Convention:**
- `firing_rate_normalized_target`: Subset matrix containing only units from selected target brain regions
- `firing_rate_normalized`: Full matrix with all units (NOT used in this section)
- `k_target`: Number of PCA components for target (k_source / 2)

**PCA Configuration:**
- Rows = units, columns = condition × time-bin features
- Components capture maximal variance (no demixing)


In [ ]:

# Determine k for target matrix: 1/2 of source matrix k
# NOTE: PCA on time × neurons (transpose) mirrors source logic
k_source = int(k_components_source)  # k from source PCA (Section 6) - ensure int
k_target = int(max(1, k_source // 2))  # Target k is 1/2 of source k (minimum 1) - ensure int

# Build time-by-neuron matrix for target
X_target = firing_rate_normalized_target.T  # shape: (total_time_bins, n_neurons_target)
T_all_target, n_neurons_target = X_target.shape

# Check constraint: n_components <= min(n_samples, n_features)
max_k_target = min(T_all_target, n_neurons_target)

# Cap k_target at max allowed
if k_target > max_k_target:
    print(f"Warning: k_target={k_target} exceeds max allowed ({max_k_target}). Capping to {max_k_target}.")
    k_target = max_k_target

print("=== Target Matrix PCA Setup ===")
print(f"Source matrix k (from Section 6): {k_source}")
print(f"Target matrix k: {k_target} (1/2 of source k, capped at max)")
print(f"Target matrix shape (time bins × neurons): {X_target.shape}")
print(f"  Time bins: {T_all_target}, Neurons: {n_neurons_target}")
print(f"  Max k allowed: {max_k_target}")


In [ ]:

# Run PCA on target matrix with k_target components (time × neuron PCA)
print(f"Running PCA on target matrix with k_target={k_target} components...")

# Run PCA on full matrix with k_target components
pca_target_full = compute_pca(X_target, n_components=k_target)

# Components: neuron weights (k_target × n_neurons_target)
principal_components_target = pca_target_full['components']

# Scores: latent activity over time (time bins × k_target)
scores_target = transform_pca(X_target, pca_target_full)

# Use latent time series (k_target × time bins) for downstream regression/null-space
scores_target_full = scores_target.T

print("=== Target Matrix PCA Results ===")
print(f"Neuron-weight matrix shape: {principal_components_target.shape} (k_target × n_neurons_target)")
print(f"Scores shape: {scores_target.shape} (time bins × k_target)")
print(f"Latent time series shape: {scores_target_full.shape} (k_target × time bins)")

print("Explained variance (k_target components):")
print(f"  Total (k_target components): {pca_target_full['explained_variance_ratio'].sum():.4f} ({pca_target_full['explained_variance_ratio'].sum()*100:.2f}%)")
print(f"  Per component:")
for i in range(min(k_target, 10)):
    print(f"    PC{i+1}: {pca_target_full['explained_variance_ratio'][i]:.4f} "
          f"({pca_target_full['explained_variance_ratio'][i]*100:.2f}%)")

print(f"✓ Target matrix PCA completed: shape ({k_target}, {scores_target_full.shape[1]})")


In [ ]:

# Store target PCA results
print("=== Stored Target PCA Results ===")
print(f"  k_target: {k_target}")
print(f"  Latent time series (pc_matrix_target) shape: {scores_target_full.shape}")
print(f"  Scores matrix (time × k): {scores_target.shape}")
print(f"  Neuron-weight matrix: shape {principal_components_target.shape}")
print(f"  Explained variance ratio: {pca_target_full['explained_variance_ratio']}")

# Store for later use
pca_model_target = pca_target_full  # Store PCA model (time × neuron)
k_components_target = k_target
pc_matrix_target = scores_target_full  # (k_target, time bins) - latent activity for downstream
score_matrix_target = scores_target  # (time bins, k_target)

print("✓ Target PCA results stored")
print(f"  Components span full matrix (all epochs)")
print(f"  Comparison: Source k={k_source}, Target k={k_target} (target = source / 2)")


## 8. Ridge Regression for Null Space Estimation

Estimate the mapping matrix W from regression epoch to projection epoch PCA components using ridge regression, trained only on regression epoch columns.


In [ ]:

# Step 1: Verify Regression/Projection Index Vectors
# Use existing idx_regression (regression epoch) and idx_project (projection epoch) from Section 4

print("=== Regression/Projection Index Vectors ===")
print(f"Total columns in combined matrix: {firing_rate_combined.shape[1]}")
print(f"  Regression epoch (idx_regression): {idx_regression.sum()} columns")
print(f"  Projection epoch (idx_project): {idx_project.sum()} columns")
print(f"Verification:")
print(f"  idx_project + idx_regression should equal all True: {(idx_project | idx_regression).all()}")
print(f"  No overlap: {(idx_project & idx_regression).sum() == 0}")
print(f"✓ Index vectors ready:")
print(f"  idx_regression: regression epoch columns")
print(f"  idx_project: projection epoch columns")


In [ ]:

# Step 2: Extract Training Matrices
# Training uses only regression epoch columns

# Extract SOURCE latent trajectories (k_source × time bins)
N_train = pc_matrix_source[:, idx_regression]

# Extract TARGET latent trajectories (k_target × time bins)
M_train = pc_matrix_target[:, idx_regression]

print("=== Training Matrices ===")
print(f"N_train (SOURCE latent, regression epoch columns): shape {N_train.shape}")
print(f"  k_source: {N_train.shape[0]} (should be {k_components_source})")
print(f"  num_regression_cols: {N_train.shape[1]}")
print(f"M_train (TARGET latent, regression epoch columns): shape {M_train.shape}")
print(f"  k_target: {M_train.shape[0]} (should be {k_components_target})")
print(f"  num_regression_cols: {M_train.shape[1]}")

print(f"✓ Training matrices extracted (using regression epoch columns only)")


In [ ]:
# Step 3: Implement Ridge Regression
# Ridge regression: M ≈ W * N
# W = (M_train @ N_train.T) @ inv(N_train @ N_train.T + lam * I_k_source)
# Shape: (k_target, k_source)

# Set ridge regularization parameter
lam = 0.1  # Can be adjusted/tuned

print("=== Ridge Regression Setup ===")
print(f"Ridge parameter λ: {lam}")
print(f"N_train shape: {N_train.shape} (k_source × num_fit_cols)")
print(f"M_train shape: {M_train.shape} (k_target × num_fit_cols)")

# Compute components of ridge regression formula
# N_train @ N_train.T: shape (k_source, k_source)
NNT = N_train @ N_train.T

# Identity matrix scaled by lambda: λ * I_k_source
I_k = lam * np.eye(N_train.shape[0])  # N_train.shape[0] = k_source

# Regularized covariance: N_train @ N_train.T + λ * I_k_source
regularized_cov = NNT + I_k

# Compute inverse: inv(N_train @ N_train.T + λ * I_k_source)
regularized_cov_inv = np.linalg.inv(regularized_cov)

# M_train @ N_train.T: shape (k_target, k_source)
MNT = M_train @ N_train.T

# Final W: shape (k_target, k_source)
W = (MNT @ regularized_cov_inv)

print(f"\n=== Ridge Regression Results ===")
print(f"W shape: {W.shape} (k_target × k_source)")
print(f"  k_target: {W.shape[0]}")
print(f"  k_source: {W.shape[1]}")
print(f"\nW statistics:")
print(f"  Mean: {W.mean():.6f}")
print(f"  Std: {W.std():.6f}")
print(f"  Min: {W.min():.6f}")
print(f"  Max: {W.max():.6f}")
print(f"\n✓ Ridge regression completed")


In [ ]:
# Step 4: Store and Verify Results

# Store W matrix and ridge parameter
W_matrix = W
lambda_ridge_value = lam

print("=== Stored Ridge Regression Results ===")
print(f"W_matrix shape: {W_matrix.shape} (k_target × k_source)")
print(f"  k_target: {W_matrix.shape[0]}")
print(f"  k_source: {W_matrix.shape[1]}")
print(f"\nLambda (ridge parameter): {lambda_ridge_value}")
print(f"\nW_matrix statistics:")
print(f"  Mean: {W_matrix.mean():.6f}")
print(f"  Std: {W_matrix.std():.6f}")
print(f"  Min: {W_matrix.min():.6f}")
print(f"  Max: {W_matrix.max():.6f}")

# Verify dimensions
assert W_matrix.shape == (k_components_target, k_components_source), \
    f"W_matrix shape mismatch: expected ({k_components_target}, {k_components_source}), got {W_matrix.shape}"

print(f"\n✓ W matrix stored and verified")
print(f"  Ready for null space projection")


## 9. SVD Decomposition for Potent/Null Spaces

Decompose W using SVD to identify potent and null spaces in source space. The potent basis spans the dimensions that map from target to source, while the null space is the orthogonal complement in R^k_source.


In [ ]:
# Step 1: Perform SVD on W
# W = U @ diag(S) @ Vt

print("=== SVD Decomposition of W ===")
print(f"W shape: {W.shape} (k_target × k_source)")

# Perform SVD
U, S, Vt = np.linalg.svd(W, full_matrices=False)

print(f"\nSVD Results:")
print(f"  U shape: {U.shape} (k_target × rank)")
print(f"  S length: {len(S)}")
print(f"  Vt shape: {Vt.shape} (rank × k_source)")

# Identify non-zero singular values (with tolerance for numerical precision)
tolerance = 1e-8
rank = (S > tolerance).sum()

print(f"\nSingular values:")
print(f"  Total: {len(S)}")
print(f"  Non-zero (rank): {rank} (tolerance: {tolerance})")
print(f"  Range: [{S.min():.6e}, {S.max():.6e}]")
print(f"\nNon-zero singular values:")
for i in range(min(rank, 10)):
    print(f"  σ{i+1}: {S[i]:.6e}")
if rank > 10:
    print(f"  ... ({rank - 10} more)")

print(f"\n✓ SVD decomposition completed")
print(f"  Rank: {rank} (≤ k_source = {N_train.shape[0]})")


In [ ]:
# Step 2: Extract Potent Basis and Compute Projection Matrices
# Potent basis in source space = rows of V^T for non-zero singular values

# Extract potent basis: V_pot = Vt[:rank, :] in source space
V_pot = Vt[:rank, :]  # Shape: (rank, k_source) in source space

# Compute null dimension
null_dim = N_train.shape[0] - rank  # N_train.shape[0] = k_source

print("=== Potent Basis ===")
print(f"V_pot shape: {V_pot.shape} (rank × k_source)")
print(f"  rank: {rank}")
print(f"  k_source: {V_pot.shape[1]}")
print(f"\nNull dimension:")
print(f"  null_dim = N_train.shape[0] - rank = {N_train.shape[0]} - {rank} = {null_dim}")
print(f"  Expected: null_dim ≥ 0")

# Compute projection matrices
# P_pot: projection onto potent space (k_source × k_source)
P_pot = V_pot.T @ V_pot  # Shape: (k_source, k_source)

# P_null: projection onto null space (k_source × k_source)
P_null = np.eye(N_train.shape[0]) - P_pot  # Shape: (k_source, k_source)

print(f"\n=== Projection Matrices ===")
print(f"P_pot shape: {P_pot.shape} (k_source × k_source)")
print(f"P_null shape: {P_null.shape} (k_source × k_source)")

print(f"\n✓ Potent basis and projection matrices computed")
print(f"  Rank: {rank} (≤ k_source = {N_train.shape[0]})")
print(f"  Null dimension: {null_dim}")


In [ ]:
# Step 3: Verify Projection Matrices

print("=== Verification ===")

# Verify P_pot properties
print(f"P_pot properties:")
print(f"  Shape: {P_pot.shape}")
print(f"  Trace (should equal rank): {np.trace(P_pot):.6f} (rank = {rank})")
print(f"  Symmetric check: {np.allclose(P_pot, P_pot.T)}")

# Verify P_null properties
print(f"\nP_null properties:")
print(f"  Shape: {P_null.shape}")
print(f"  Trace (should equal null_dim): {np.trace(P_null):.6f} (null_dim = {null_dim})")
print(f"  Symmetric check: {np.allclose(P_null, P_null.T)}")

# Verify orthogonality: P_pot @ P_null should be approximately zero
orthogonality_check = P_pot @ P_null
max_orthogonality_error = np.abs(orthogonality_check).max()
print(f"\nOrthogonality check (P_pot @ P_null):")
print(f"  Max absolute value: {max_orthogonality_error:.6e}")
print(f"  Should be ≈ 0: {'✓' if max_orthogonality_error < 1e-10 else '✗'}")

# Verify completeness: P_pot + P_null should equal identity
completeness_check = P_pot + P_null
identity_error = np.abs(completeness_check - np.eye(N_train.shape[0])).max()
print(f"\nCompleteness check (P_pot + P_null = I):")
print(f"  Max absolute error: {identity_error:.6e}")
print(f"  Should be ≈ 0: {'✓' if identity_error < 1e-10 else '✗'}")

print(f"\n✓ Projection matrices verified")


In [ ]:
# Step 4: Store Results

# Store SVD components and projection matrices
U_svd = U
S_svd = S
Vt_svd = Vt
V_pot_stored = V_pot
P_pot_stored = P_pot
P_null_stored = P_null
rank_potent = rank
dim_null_stored = null_dim

print("=== Stored SVD Results ===")
print(f"U_svd shape: {U_svd.shape} (k_target × rank)")
print(f"S_svd length: {len(S_svd)}")
print(f"Vt_svd shape: {Vt_svd.shape} (rank × k_source)")
print(f"\nV_pot shape: {V_pot_stored.shape} (rank × k_source)")
print(f"P_pot shape: {P_pot_stored.shape} (k_source × k_source)")
print(f"P_null shape: {P_null_stored.shape} (k_source × k_source)")
print(f"\nrank_potent: {rank_potent}")
print(f"dim_null: {dim_null_stored}")

# Verify dimensions
dim_check = rank_potent + dim_null_stored == N_train.shape[0]
print(f"\nDimension check:")
print(f"  rank_potent + dim_null = {rank_potent} + {dim_null_stored} = {rank_potent + dim_null_stored}")
print(f"  k_source = {N_train.shape[0]}")
print(f"  Match: {'✓' if dim_check else '✗'}")

# Verify reconstruction: W ≈ U @ diag(S) @ Vt
W_reconstructed = U @ np.diag(S) @ Vt
reconstruction_error = np.abs(W - W_reconstructed).max()
print(f"\nReconstruction check (W ≈ U @ diag(S) @ Vt):")
print(f"  Max absolute error: {reconstruction_error:.6e}")
print(f"  Should be ≈ 0: {'✓' if reconstruction_error < 1e-10 else '✗'}")

print(f"\n✓ SVD decomposition completed and verified")
print(f"  Potent space: rank {rank_potent} (in R^k_source = R^{N_train.shape[0]})")
print(f"  Null space: dimension {dim_null_stored} (in R^k_source = R^{N_train.shape[0]})")


## 10. Projection to All Columns

Project source PCA components for ALL columns (encoding/delay/probe/press) onto the potent and null spaces using P_pot and P_null. This decomposes the entire source activity into components that map to target (potent) and components that do not (null).


In [ ]:
# Step 1: Extract Source PCA Components for All Columns
# N_all: (k_source, T_all) source PCs for any columns (encoding/delay/probe/press)

# Extract source PCA components for all columns
N_all = pc_matrix_source  # Shape: (k_source, T_all) where T_all = total number of time bins
T_all = N_all.shape[1]

print("=== Source PCA Components for All Columns ===")
print(f"N_all shape: {N_all.shape} (k_source × T_all)")
print(f"  k_source: {N_all.shape[0]}")
print(f"  T_all (total columns): {T_all}")
print(f"\nEpoch breakdown:")
print(f"  Projection epoch (idx_project): {idx_project.sum()} columns")
print(f"  Regression epoch (idx_regression): {idx_regression.sum()} columns")
print(f"  Total: {idx_project.sum() + idx_regression.sum()} columns")

print(f"\n✓ N_all extracted: source PCA components for all time bins")


In [ ]:
# Step 2: Project onto Potent Space
# N_pot = P_pot @ N_all

print("=== Projection onto Potent Space ===")
print(f"P_pot shape: {P_pot.shape} (k_source × k_source)")
print(f"N_all shape: {N_all.shape} (k_source × T_all)")

# Project N_all onto potent space
N_pot = P_pot @ N_all  # Shape: (k_source, T_all)

print(f"\nN_pot shape: {N_pot.shape} (k_source × T_all)")
print(f"  k_source: {N_pot.shape[0]}")
print(f"  T_all: {N_pot.shape[1]}")

print(f"\nN_pot statistics:")
print(f"  Mean: {N_pot.mean():.6f}")
print(f"  Std: {N_pot.std():.6f}")
print(f"  Min: {N_pot.min():.6f}")
print(f"  Max: {N_pot.max():.6f}")

print(f"\n✓ Potent projection completed")
print(f"  N_pot represents component that maps to target (regression epoch)")


In [ ]:
# Step 3: Project onto Null Space
# N_null = P_null @ N_all

print("=== Projection onto Null Space ===")
print(f"P_null shape: {P_null.shape} (k_source × k_source)")
print(f"N_all shape: {N_all.shape} (k_source × T_all)")

# Project N_all onto null space
N_null = P_null @ N_all  # Shape: (k_source, T_all)

print(f"\nN_null shape: {N_null.shape} (k_source × T_all)")
print(f"  k_source: {N_null.shape[0]}")
print(f"  T_all: {N_null.shape[1]}")

print(f"\nN_null statistics:")
print(f"  Mean: {N_null.mean():.6f}")
print(f"  Std: {N_null.std():.6f}")
print(f"  Min: {N_null.min():.6f}")
print(f"  Max: {N_null.max():.6f}")

print(f"\n✓ Null projection completed")
print(f"  N_null represents component that does NOT map to target")


In [ ]:
# Step 4: Verify Decomposition

print("=== Verification ===")

# Verify that potent + null = original
N_all_reconstructed = N_pot + N_null
reconstruction_error = np.abs(N_all - N_all_reconstructed).max()
print(f"Reconstruction check (N_pot + N_null = N_all):")
print(f"  Max absolute error: {reconstruction_error:.6e}")
print(f"  Should be ≈ 0: {'✓' if reconstruction_error < 1e-10 else '✗'}")

# Verify orthogonality: N_pot.T @ N_null should be approximately zero
orthogonality_matrix = N_pot.T @ N_null  # Shape: (T_all, T_all)
max_orthogonality_error = np.abs(orthogonality_matrix).max()
print(f"\nOrthogonality check (N_pot.T @ N_null ≈ 0):")
print(f"  Max absolute value: {max_orthogonality_error:.6e}")
print(f"  Should be ≈ 0: {'✓' if max_orthogonality_error < 1e-10 else '✗'}")

# Compute variance explained by each component
variance_all = np.var(N_all, axis=0).sum()  # Total variance
variance_pot = np.var(N_pot, axis=0).sum()  # Variance in potent space
variance_null = np.var(N_null, axis=0).sum()  # Variance in null space
variance_ratio = variance_pot / variance_null if variance_null > 0 else np.inf

print(f"\nVariance explained:")
print(f"  Total variance (N_all): {variance_all:.6f}")
print(f"  Potent variance (N_pot): {variance_pot:.6f} ({variance_pot/variance_all*100:.2f}%)")
print(f"  Null variance (N_null): {variance_null:.6f} ({variance_null/variance_all*100:.2f}%)")
print(f"  Potent/Null ratio: {variance_ratio:.4f}")

# Verify variance decomposition (should be approximately equal due to orthogonality)
variance_sum = variance_pot + variance_null
variance_decomp_error = np.abs(variance_all - variance_sum) / variance_all
print(f"\nVariance decomposition check:")
print(f"  variance_pot + variance_null = {variance_sum:.6f}")
print(f"  Relative error: {variance_decomp_error:.6e}")
print(f"  Should be ≈ 0: {'✓' if variance_decomp_error < 1e-6 else '✗'}")

print(f"\n✓ Decomposition verified")


In [ ]:
# Step 5: Store Results

# Store all projection results
N_all_stored = N_all
N_pot_stored = N_pot
N_null_stored = N_null

# Store variance statistics
projection_stats = {
    'variance_all': variance_all,
    'variance_pot': variance_pot,
    'variance_null': variance_null,
    'variance_ratio': variance_ratio,
    'reconstruction_error': reconstruction_error,
    'orthogonality_error': max_orthogonality_error
}

print("=== Stored Projection Results ===")
print(f"N_all_stored shape: {N_all_stored.shape} (k_source × T_all)")
print(f"N_pot_stored shape: {N_pot_stored.shape} (k_source × T_all)")
print(f"N_null_stored shape: {N_null_stored.shape} (k_source × T_all)")
print(f"\nProjection statistics:")
print(f"  Variance ratio (pot/null): {projection_stats['variance_ratio']:.4f}")
print(f"  Reconstruction error: {projection_stats['reconstruction_error']:.6e}")
print(f"  Orthogonality error: {projection_stats['orthogonality_error']:.6e}")

print(f"\n✓ All projection results stored")
print(f"  Ready for further analysis and visualization")


## 11. Per-Epoch Null vs Potent Occupancy (Tuning Ratio)

Compute tuning ratio (TR) for each epoch by measuring dimension-normalized energy in potent vs null spaces. Expected: TR > 1 in projection epoch (more null space activity), TR decreases in regression epoch (more potent space activity). Include bootstrap confidence intervals.


In [ ]:
# Step 1: Create Epoch Column Masks
# Currently we have idx_project (projection epoch) and idx_regression (regression epoch)

print("=== Epoch Column Masks ===")
print(f"Total columns: {N_all.shape[1]}")

print(f"\nidx_project (projection epoch):")
print(f"  Number of columns: {idx_project.sum()}")
print(f"  Column indices: {np.where(idx_project)[0][:10]}..." if idx_project.sum() > 10 else f"  Column indices: {np.where(idx_project)[0]}")

print(f"\nidx_regression (regression epoch):")
print(f"  Number of columns: {idx_regression.sum()}")
print(f"  Column indices: {np.where(idx_regression)[0][:10]}..." if idx_regression.sum() > 10 else f"  Column indices: {np.where(idx_regression)[0]}")

# Store epoch masks
epoch_masks = {
    'projection': idx_project,  # Projection epoch
    'regression': idx_regression       # Regression epoch
}

print(f"\n✓ Epoch masks created:")
print(f"  projection: {epoch_masks['projection'].sum()} columns")
print(f"  regression: {epoch_masks['regression'].sum()} columns")


In [ ]:
# Step 2: Compute Energy Functions
# E_pot(E) = (1/r) * <||P_pot @ N_E||_F^2>
# E_null(E) = (1/(k-r)) * <||P_null @ N_E||_F^2>

# Get dimensions
r = rank_potent  # Rank of potent space
k = N_all.shape[0]  # k_source

print("=== Energy Computation Setup ===")
print(f"r (rank_potent): {r}")
print(f"k (k_source): {k}")
print(f"k - r (null_dim): {k - r}")

def compute_energy(epoch_mask):
    """
    Compute dimension-normalized energy in potent and null spaces for an epoch.
    
    Args:
        epoch_mask: Boolean array indicating columns belonging to epoch
        
    Returns:
        E_pot: Dimension-normalized energy in potent space
        E_null: Dimension-normalized energy in null space
    """
    # Extract source PCs for this epoch
    N_E = N_all[:, epoch_mask]  # Shape: (k_source, num_epoch_cols)
    
    if N_E.shape[1] == 0:
        return np.nan, np.nan
    
    # Project onto potent space: P_pot @ N_E
    N_E_pot = P_pot @ N_E  # Shape: (k_source, num_epoch_cols)
    
    # Project onto null space: P_null @ N_E
    N_E_null = P_null @ N_E  # Shape: (k_source, num_epoch_cols)
    
    # Compute squared Frobenius norm for each column, then take mean
    # ||A||_F^2 = sum of squares of all elements
    # For each column: ||N_E_pot[:, i]||^2
    frobenius_norm_sq_pot = np.sum(N_E_pot ** 2, axis=0)  # Shape: (num_epoch_cols,)
    frobenius_norm_sq_null = np.sum(N_E_null ** 2, axis=0)  # Shape: (num_epoch_cols,)
    
    # Mean across columns: <||P_pot @ N_E||_F^2>
    mean_frob_sq_pot = np.mean(frobenius_norm_sq_pot)
    mean_frob_sq_null = np.mean(frobenius_norm_sq_null)
    
    # Dimension-normalized energy
    E_pot = (1.0 / r) * mean_frob_sq_pot if r > 0 else np.nan
    E_null = (1.0 / (k - r)) * mean_frob_sq_null if (k - r) > 0 else np.nan
    
    return E_pot, E_null

print(f"\n✓ Energy computation function defined")
print(f"  E_pot(E) = (1/r) * mean(||P_pot @ N_E||_F^2)")
print(f"  E_null(E) = (1/(k-r)) * mean(||P_null @ N_E||_F^2)")


In [ ]:
# Step 3: Compute Tuning Ratio for Each Epoch

print("=== Tuning Ratio Computation ===")

epoch_results = {}

# Compute energy and TR for each epoch
for epoch_name, epoch_mask in epoch_masks.items():
    E_pot_val, E_null_val = compute_energy(epoch_mask)
    
    # Compute tuning ratio: TR = E_null / E_pot
    if E_pot_val > 0 and not np.isnan(E_pot_val) and not np.isnan(E_null_val):
        TR = E_null_val / E_pot_val
    else:
        TR = np.nan
    
    epoch_results[epoch_name] = {
        'E_pot': E_pot_val,
        'E_null': E_null_val,
        'TR': TR,
        'n_columns': epoch_mask.sum()
    }
    
    print(f"\n{epoch_name.upper()} epoch:")
    print(f"  Columns: {epoch_mask.sum()}")
    print(f"  E_pot: {E_pot_val:.6f}")
    print(f"  E_null: {E_null_val:.6f}")
    print(f"  TR = E_null/E_pot: {TR:.4f}")

# Create summary DataFrame
import pandas as pd
results_df = pd.DataFrame(epoch_results).T
results_df.index.name = 'epoch'
results_df = results_df.reset_index()

print(f"\n=== Summary Table ===")
print(results_df.to_string(index=False))

print(f"\n✓ Tuning ratios computed")
print(f"  Expected: TR > 1 in projection epoch (more null space activity)")
print(f"  Expected: TR decreases in regression epoch (more potent space activity)")


In [ ]:
# Step 4: Bootstrap Confidence Intervals

print("=== Bootstrap Confidence Intervals ===")

n_bootstrap = 1000  # Number of bootstrap iterations
bootstrap_results = {}

for epoch_name, epoch_mask in epoch_masks.items():
    # Get column indices for this epoch
    epoch_col_indices = np.where(epoch_mask)[0]
    n_cols = len(epoch_col_indices)
    
    if n_cols == 0:
        bootstrap_results[epoch_name] = {
            'TR_samples': [],
            'TR_mean': np.nan,
            'TR_ci_lower': np.nan,
            'TR_ci_upper': np.nan
        }
        continue
    
    # Bootstrap: resample columns with replacement
    np.random.seed(42)  # For reproducibility
    TR_samples = []
    
    for i in range(n_bootstrap):
        # Resample column indices with replacement
        bootstrap_indices = np.random.choice(epoch_col_indices, size=n_cols, replace=True)
        
        # Create bootstrap mask
        bootstrap_mask = np.zeros(N_all.shape[1], dtype=bool)
        bootstrap_mask[bootstrap_indices] = True
        
        # Compute energy and TR for bootstrap sample
        E_pot_boot, E_null_boot = compute_energy(bootstrap_mask)
        
        if E_pot_boot > 0 and not np.isnan(E_pot_boot) and not np.isnan(E_null_boot):
            TR_boot = E_null_boot / E_pot_boot
            TR_samples.append(TR_boot)
    
    # Compute statistics
    TR_samples = np.array(TR_samples)
    TR_mean = np.mean(TR_samples)
    TR_ci_lower = np.percentile(TR_samples, 2.5)  # 95% CI lower bound
    TR_ci_upper = np.percentile(TR_samples, 97.5)  # 95% CI upper bound
    
    bootstrap_results[epoch_name] = {
        'TR_samples': TR_samples,
        'TR_mean': TR_mean,
        'TR_ci_lower': TR_ci_lower,
        'TR_ci_upper': TR_ci_upper
    }
    
    print(f"\n{epoch_name.upper()} epoch bootstrap:")
    print(f"  Mean TR: {TR_mean:.4f}")
    print(f"  95% CI: [{TR_ci_lower:.4f}, {TR_ci_upper:.4f}]")
    print(f"  Original TR: {epoch_results[epoch_name]['TR']:.4f}")

print(f"\n✓ Bootstrap confidence intervals computed ({n_bootstrap} iterations)")


In [ ]:
# Step 5: Visualize Results

print("=== Visualization ===")

# Prepare data for plotting
epoch_names = list(epoch_masks.keys())
TR_values = [epoch_results[epoch]['TR'] for epoch in epoch_names]
TR_means = [bootstrap_results[epoch]['TR_mean'] for epoch in epoch_names]
TR_ci_lower = [bootstrap_results[epoch]['TR_ci_lower'] for epoch in epoch_names]
TR_ci_upper = [bootstrap_results[epoch]['TR_ci_upper'] for epoch in epoch_names]

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Plot bars with error bars
x_pos = np.arange(len(epoch_names))
bars = ax.bar(x_pos, TR_means, yerr=[np.array(TR_means) - np.array(TR_ci_lower), 
                                       np.array(TR_ci_upper) - np.array(TR_means)],
              capsize=5, alpha=0.7, color=['steelblue', 'coral'])

# Add horizontal line at TR=1
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1, label='TR = 1')

# Customize plot
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Tuning Ratio (TR = E_null / E_pot)', fontsize=12)
ax.set_title('Per-Epoch Tuning Ratio\n(TR > 1: more null space, TR < 1: more potent space)', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(epoch_names)
ax.legend()
ax.grid(True, alpha=0.3)

# Add value labels on bars
for i, (mean, lower, upper) in enumerate(zip(TR_means, TR_ci_lower, TR_ci_upper)):
    ax.text(i, mean + (upper - mean) + 0.05, f'{mean:.3f}', 
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n✓ Visualization created")
print(f"  Expected pattern: TR > 1 in projection epoch, TR decreases in regression epoch")


In [ ]:
# Step 6: Store Results

# Store all results
epoch_masks_stored = epoch_masks.copy()
tuning_ratio_results = {
    'epoch_results': epoch_results,
    'bootstrap_results': bootstrap_results,
    'results_df': results_df
}

print("=== Stored Tuning Ratio Results ===")
print(f"Epoch masks stored: {list(epoch_masks_stored.keys())}")
print(f"Epoch results stored: {list(epoch_results.keys())}")
print(f"Bootstrap results stored: {list(bootstrap_results.keys())}")

print(f"\n=== Final Summary Table ===")
summary_df = results_df.copy()
summary_df['TR_mean_boot'] = [bootstrap_results[epoch]['TR_mean'] for epoch in summary_df['epoch']]
summary_df['TR_ci_lower'] = [bootstrap_results[epoch]['TR_ci_lower'] for epoch in summary_df['epoch']]
summary_df['TR_ci_upper'] = [bootstrap_results[epoch]['TR_ci_upper'] for epoch in summary_df['epoch']]
print(summary_df.to_string(index=False))

print(f"\n✓ All tuning ratio results stored")
print(f"  Ready for further analysis and across-subjects comparison")
